Mismo test que `clients/terminal/find-user-by-name.txt`.


In [ ]:
import os
import subprocess
from pathlib import Path

from IPython import get_ipython

app_root = Path.cwd().resolve()
if app_root.name == "jupyter" and app_root.parent.name == "clients":
    app_root = app_root.parent.parent
os.chdir(app_root)

_ip = get_ipython()


def _node_input_transformer(lines):
    code = "".join(lines).strip()
    if not code.startswith("node --input-type=module"):
        return lines
    return [
        "import subprocess\n",
        f"_c = subprocess.run({code!r}, shell=True, cwd={str(app_root)!r}, "
        "capture_output=True, text=True, encoding='utf-8', errors='replace')\n",
        "print(_c.stdout, end='') if _c.stdout else None\n",
        "print(_c.stderr, end='') if _c.stderr else None\n",
    ]


if _ip is not None and not getattr(_ip, "_first_app_node_hook", False):
    _ip.input_transformers_cleanup.append(_node_input_transformer)
    _ip._first_app_node_hook = True

print("cwd:", os.getcwd())


Test 1: dos usuarios mismo nombre (OK).


In [ ]:
node --input-type=module -e "import { createUser } from './logic/create-user.js'; import { findUserByName } from './logic/find-user-by-name.js'; try { const stamp = Date.now(); createUser(['Ada', 'ada-name-' + stamp + '@mail.com']); createUser(['Ada', 'ada-name2-' + stamp + '@mail.com']); console.log(findUserByName('Ada')); } catch (error) { console.log(error.message); }"


Test 2: nombre vacío (error).


In [ ]:
node --input-type=module -e "import { findUserByName } from './logic/find-user-by-name.js'; try { findUserByName('   '); } catch (error) { console.log(error.message); }"


Test 3: nombre sin coincidencias (lista vacía).


In [ ]:
node --input-type=module -e "import { findUserByName } from './logic/find-user-by-name.js'; try { console.log(findUserByName('NombreQueNoExiste742')); } catch (error) { console.log(error.message); }"
